In [ ]:
# 自动评估
# % cd cot/distill/GPU_run/oai/
!pip3 install openai == 0.28
import openai
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import json
import random
import re
from typing import Literal
from pydantic import BaseModel

class ExtractedAnswer(BaseModel):
    extracted_final_answer: str
    reasoning: str
    correct: Literal["yes", "no"]
acc = 0
all_num = 0   
JUDGE_PROMPT = """Judge whether the following [response] to [question] is correct or not based on the precise and unambiguous [correct_answer] below.

[question]: {question}

[response]: {response}

Your judgement must be in the format and criteria specified below:

extracted_final_answer: The final exact answer extracted from the [response]. Put the extracted answer as 'None' if there is no exact, final answer to extract from the response.

[correct_answer]: {correct_answer}

reasoning: Explain why the extracted_final_answer is correct or incorrect based on [correct_answer], focusing only on if there are meaningful differences between [correct_answer] and the extracted_final_answer. Do not comment on any background to the problem, do not attempt to solve the problem, do not argue for any answer different than [correct_answer], focus only on whether the answers match.

correct: Answer 'yes' if extracted_final_answer matches the [correct_answer] given above, or is within a small margin of error for numerical problems. Answer 'no' otherwise, i.e. if there if there is any inconsistency, ambiguity, non-equivalency, or if the extracted answer is incorrect.
"""

openai.api_key="your key here"
# openai.api_base=""

data = []
with open("../distill/GPU_run/oai_judge.jsonl", "r", encoding="utf-8") as f:

    for line in f:
        data.append(json.loads(line.strip()))
sample_data = data

for ele in sample_data:
    
    prompt = JUDGE_PROMPT.format(question=ele["question"], correct_answer=ele["answer"], response=ele["d_final_cot"])
    response = openai.ChatCompletion.create(
        model="gpt-4.1",  # 或者使用 "gpt-4"
        messages=[
            # {"role": "system", "content": "You are a helpful assistant who can solve the problem step by step. Try to solve it in serveral steps and detailly show reflection and progression in your answers"},
           {"role": "user", "content": prompt}
        ],
        # response_format={"type": "json_object"} 
    )
    content = response.choices[0].message['content']
    matches = re.findall(r"(?:Correct:|correct:)\s*(\w+)", content)
    ele["final_ACC"] = "no"
    if matches:
        for single_answer in matches:
            if "yes" in single_answer or "Yes" in single_answer or "YES" in single_answer:
                acc+=1
                ele["final_ACC"] = "yes"
                break
    all_num+=1
    print(content)
    print("---")
    
    if all_num == 100:
        print(f"------------------------ acc {acc}")
        
print("acc",acc,"all_num",all_num)
# print(sample_data[-1]["token_num"]*len(sample_data))
# with open("./best/final_g_mmlu_judge_9540_2.5_0.5_new.jsonl","w",encoding = "utf8") as fw:
#     for ele in sample_data:
#         fw.write(json.dumps(ele,ensure_ascii=False)+"\n")
    



ERROR: Invalid requirement: '=='
extracted_final_answer: -1

reasoning: The extracted final answer \(-1\) exactly matches the correct_answer \(-1\), with no inconsistency, ambiguity, or error.

correct: yes
---
extracted_final_answer: \boxed{\dfrac{9}{100}}

reasoning: The extracted_final_answer exactly matches the [correct_answer] \frac{9}{100}, with the boxed formatting not affecting the mathematical value. There are no inconsistencies, ambiguities, or errors in the final answer.

correct: yes
---
extracted_final_answer: [-2, 7]

reasoning: The extracted_final_answer, [-2, 7], matches exactly with the correct_answer, x ∈ [-2,7], as both specify the same closed interval for x in interval notation.

correct: yes
---
extracted_final_answer: 46

reasoning: The extracted_final_answer is 46, which matches exactly with the [correct_answer] of 46. There is no inconsistency or ambiguity between the two answers.

correct: yes
---
extracted_final_answer: 0

reasoning: The extracted final answer